# Trash Classifier - Training Notebook

Classify trash images into 3 classes: **paper / glass / plastic**.

We use a classic computer vision approach (no deep learning):
- Features: color histogram (BGR + HSV) + LBP + GLCM + HOG
- Classifier: SVM with RBF kernel, tuned with 5-fold GridSearchCV

Run this whole notebook top to bottom to train and save the model.

## 1. Imports

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops, hog

from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

## 2. Settings

The dataset folders `paper/`, `glass/`, `plastic/` should live in the project root next to this notebook.

In [ ]:
IMG_SIZE = 224
CLASSES = ["paper", "glass", "plastic"]
DATA_DIR = "."
MODELS_DIR = "models"
RANDOM_STATE = 42

os.makedirs(MODELS_DIR, exist_ok=True)

## 3. Preprocessing helpers

Just simple image loading + resize + color conversions.

In [ ]:
def load_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError("Cannot read image: " + path)
    return img

def resize_img(img, size=IMG_SIZE):
    return cv2.resize(img, (size, size))

def to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def to_hsv(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

## 4. Feature extraction

Four feature blocks, concatenated into one long vector per image:
- **Color histogram** (BGR + HSV, 16 bins per channel)
- **LBP** (Local Binary Pattern, P=24 R=3, uniform)
- **GLCM** (quantized to 8 levels, 5 props x 2 distances x 4 angles)
- **HOG** (9 orientations, 16x16 cells, 2x2 blocks)

In [ ]:
def color_hist(img):
    feats = []
    # BGR
    for c in range(3):
        h = cv2.calcHist([img], [c], None, [16], [0, 256]).flatten()
        h = h / (h.sum() + 1e-7)
        feats.append(h)
    # HSV
    hsv = to_hsv(img)
    ranges = [(0, 180), (0, 256), (0, 256)]
    for c, r in enumerate(ranges):
        h = cv2.calcHist([hsv], [c], None, [16], list(r)).flatten()
        h = h / (h.sum() + 1e-7)
        feats.append(h)
    return np.concatenate(feats)

def lbp_hist(gray):
    P, R = 24, 3
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=P + 2, range=(0, P + 2))
    hist = hist.astype("float32")
    hist /= (hist.sum() + 1e-7)
    return hist

def glcm_feat(gray):
    # quantize to 8 levels so glcm is small and fast
    g = (gray // 32).astype(np.uint8)
    distances = [1, 2]
    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
    glcm = graycomatrix(g, distances=distances, angles=angles, levels=8, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]
    out = []
    for p in props:
        out.append(graycoprops(glcm, p).flatten())
    return np.concatenate(out).astype("float32")

def hog_feat(gray):
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    ).astype("float32")

def extract_features(img):
    img = resize_img(img)
    gray = to_gray(img)
    return np.concatenate([
        color_hist(img),
        lbp_hist(gray),
        glcm_feat(gray),
        hog_feat(gray),
    ]).astype("float32")

# quick sanity check on feature vector size
_dummy = np.zeros((224, 224, 3), dtype=np.uint8)
print("Feature vector size:", extract_features(_dummy).shape[0])

## 5. Augmentation (training set only)

From the proposal: flip, rotate +/-20 deg, zoom 10%, brightness shift +/-30. Each training image gets 4 extra variants, so the training set becomes 5x bigger.

In [ ]:
def augment(img, seed):
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    variants = []

    # 1. horizontal flip
    variants.append(cv2.flip(img, 1))

    # 2. rotation +/-20 deg
    angle = float(rng.uniform(-20, 20))
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    variants.append(cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT))

    # 3. zoom 10%
    factor = float(rng.uniform(0.9, 1.1))
    if factor >= 1.0:
        nh, nw = max(1, int(h / factor)), max(1, int(w / factor))
        y0, x0 = (h - nh) // 2, (w - nw) // 2
        crop = img[y0:y0 + nh, x0:x0 + nw]
        variants.append(cv2.resize(crop, (w, h)))
    else:
        sw, sh = max(1, int(w * factor)), max(1, int(h * factor))
        scaled = cv2.resize(img, (sw, sh))
        canvas = np.zeros_like(img)
        y0, x0 = (h - sh) // 2, (w - sw) // 2
        canvas[y0:y0 + sh, x0:x0 + sw] = scaled
        variants.append(canvas)

    # 4. brightness shift
    beta = int(rng.uniform(-30, 30))
    variants.append(cv2.convertScaleAbs(img, alpha=1.0, beta=beta))

    return variants

## 6. Load dataset paths

In [ ]:
paths = []
labels = []
for cls in CLASSES:
    d = Path(DATA_DIR) / cls
    if not d.is_dir():
        raise FileNotFoundError("Missing class folder: " + str(d))
    files = sorted([str(p) for p in d.iterdir() if p.suffix.lower() == ".jpg"])
    print(cls, "->", len(files), "images")
    paths.extend(files)
    labels.extend([cls] * len(files))

print("Total:", len(paths), "images")

## 7. Train / Val / Test split (70 / 15 / 15)

In [ ]:
le = LabelEncoder()
y_all = le.fit_transform(labels)
paths_arr = np.array(paths)

p_tv, p_test, y_tv, y_test_lbl = train_test_split(
    paths_arr, y_all, test_size=0.15, stratify=y_all, random_state=RANDOM_STATE
)
# split the remaining 85% into train (70%) and val (15%)
p_train, p_val, y_train_lbl, y_val_lbl = train_test_split(
    p_tv, y_tv, test_size=0.15 / 0.85, stratify=y_tv, random_state=RANDOM_STATE
)

print("train:", len(p_train), "  val:", len(p_val), "  test:", len(p_test))

## 8. Extract features for all splits

Training set is augmented (5x), val/test are plain. This is the slow step.

In [ ]:
def build_features(paths_list, labels_list, do_augment, seed_base=0):
    feats = []
    out_labels = []
    n = len(paths_list)
    t0 = time.time()
    for i, (p, lbl) in enumerate(zip(paths_list, labels_list)):
        img = load_image(p)
        feats.append(extract_features(img))
        out_labels.append(int(lbl))
        if do_augment:
            for aug_img in augment(img, seed=seed_base + i):
                feats.append(extract_features(aug_img))
                out_labels.append(int(lbl))
        if (i + 1) % 50 == 0 or (i + 1) == n:
            print("  ", i + 1, "/", n)
    print("  done in", round(time.time() - t0, 1), "s")
    return np.vstack(feats), np.asarray(out_labels)

print("Train features (with augmentation)...")
X_train, y_train = build_features(p_train, y_train_lbl, do_augment=True)

print("Val features...")
X_val, y_val = build_features(p_val, y_val_lbl, do_augment=False)

print("Test features...")
X_test, y_test = build_features(p_test, y_test_lbl, do_augment=False)

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

## 9. Train SVM (GridSearchCV)

Pipeline: `StandardScaler` -> `SVC(rbf)`. Grid: C in {1, 10, 100}, gamma in {scale, 0.001, 0.01}. 5-fold CV.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, decision_function_shape="ovo", random_state=RANDOM_STATE)),
])

param_grid = {
    "svm__C": [1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV f1_macro:", round(grid.best_score_, 4))

best_pipe = grid.best_estimator_
print("Val accuracy: ", round(best_pipe.score(X_val, y_val), 4))
print("Test accuracy:", round(best_pipe.score(X_test, y_test), 4))

## 10. Evaluate on test set

Classification report + confusion matrix.

In [ ]:
y_pred = best_pipe.predict(X_test)
target_names = list(le.classes_)

report = classification_report(y_test, y_pred, target_names=target_names, digits=4)
print(report)

# save report to a text file too
with open(os.path.join(MODELS_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 11. Inference latency

Proposal target (Section 4, Table 2): **<= 200 ms** per image (minimum), **<= 100 ms** (ideal).

We measure end-to-end latency on the test set: feature extraction + scaling + SVM predict, the same pipeline `app.py` runs on each upload.

In [ ]:
latencies = []
for p in p_test:
    img = load_image(p)
    t0 = time.perf_counter()
    feats = extract_features(img).reshape(1, -1)
    _ = best_pipe.predict(feats)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print("End-to-end inference latency on test set (ms):")
print("  mean:  ", round(latencies.mean(), 2))
print("  median:", round(np.median(latencies), 2))
print("  p95:   ", round(np.percentile(latencies, 95), 2))
print("  max:   ", round(latencies.max(), 2))
print()
print("Proposal target: <=200 ms (min), <=100 ms (ideal)")
print("Pass min target:  ", "YES" if latencies.mean() <= 200 else "NO")
print("Pass ideal target:", "YES" if latencies.mean() <= 100 else "NO")

## 12. Learning curve (Accuracy & F1-macro)

Section 4 of the proposal asks for a Loss & Accuracy Curve to monitor over/underfitting. SVMs don't produce a per-epoch loss the way neural networks do, so the standard equivalent is a **learning curve**: train and CV scores plotted against training-set size.

We plot it twice (accuracy and F1-macro) so both the accuracy curve from the proposal table and the F1-macro metric are covered. This step refits the model several times -- it can take a few minutes.

In [ ]:
for metric, fname in [("accuracy", "learning_curve_accuracy.png"),
                       ("f1_macro", "learning_curve_f1.png")]:
    sizes, train_scores, val_scores = learning_curve(
        best_pipe, X_train, y_train, cv=5,
        train_sizes=np.linspace(0.1, 1.0, 6),
        scoring=metric, n_jobs=-1, random_state=RANDOM_STATE, shuffle=True,
    )
    tm, ts = train_scores.mean(axis=1), train_scores.std(axis=1)
    vm, vs = val_scores.mean(axis=1), val_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(sizes, tm, "o-", label="Train " + metric)
    ax.fill_between(sizes, tm - ts, tm + ts, alpha=0.2)
    ax.plot(sizes, vm, "o-", label="CV " + metric)
    ax.fill_between(sizes, vm - vs, vm + vs, alpha=0.2)
    ax.set_xlabel("Training examples")
    ax.set_ylabel(metric)
    ax.set_title("Learning curve - " + metric + " (RBF SVM, 5-fold CV)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, fname), dpi=150)
    plt.show()

## 13. Save model artifacts

We save the SVM and scaler separately so the Streamlit app can load them easily.

In [ ]:
fitted_scaler = best_pipe.named_steps["scaler"]
fitted_svm = best_pipe.named_steps["svm"]

joblib.dump(fitted_svm, os.path.join(MODELS_DIR, "svm_model.joblib"))
joblib.dump(fitted_scaler, os.path.join(MODELS_DIR, "scaler.joblib"))
joblib.dump(le, os.path.join(MODELS_DIR, "label_encoder.joblib"))
joblib.dump({"X_test": X_test, "y_test": y_test}, os.path.join(MODELS_DIR, "test_split.joblib"))

print("Saved model files to", MODELS_DIR)